# 112 — CPTAC-3 RNA-seq Technical Characterization and Source QC Provenance

## Scientific objective

Characterize the audited primary-tumor STAR Counts source files from GDC CPTAC-3 and assess the availability and provenance of sequencing-quality metadata without assigning sample-level QC eligibility.

The notebook consumes source identities and acquisition-integrity evidence generated by **Notebook 111**, together with two version-controlled GDC source metadata snapshots under `docs/evidence/phase1/notebook112/`. It does not read outputs from Notebook 113 or any Phase 8 notebook.

The source-only analysis identifies uniquely represented primary RNA-seq cases, reads their local STAR `unstranded` count files, examines additional same-case RNA files, and reconstructs a bounded, deterministic 12-file source-metadata audit. It distinguishes metadata agreement from independent sequencing-quality certification.

## Downstream outputs

Exactly two generated files are persisted under `data/interim/metadata/`:

- `112_cptac3_rna_simple_primary_technical_inventory.csv`: file- and case-keyed technical inventory for 1,082 uniquely represented primary candidates.
- `112_cptac3_qc_provenance_handoff.json`: SHA-256-fingerprinted source provenance, output identity, descriptive audit results and unresolved QC limitations.

## Interpretation

File integrity and raw-count validity do **not** constitute independent read-level/library QC. No expression normalization, program scoring, TCGA lineage assignment, clinical prediction, sample-level QC exclusion, or donor-overlap inference is performed here. Frozen program-gene compatibility and proposed cohort construction belong to Notebook 113; lineage-aware inferential replication belongs to Notebook 800.

In [1]:
# =============================================================================
# Imports
# =============================================================================

import json
import platform

import numpy as np
import pandas as pd
from IPython.display import display

from pancancer_epigenetics.utils.file_checks import calculate_sha256
from pancancer_epigenetics.utils.paths import Paths, project_relative_path

In [2]:
# =============================================================================
# Input and output paths
# =============================================================================

ROOT = Paths.root
META = Paths.metadata

AUDIT_111_PATH = META / "111_cptac3_acquisition_audit_summary.json"
FILE_INDEX_111_PATH = META / "111_cptac3_file_index.csv"
PRIMARY_111_PATH = META / "111_cptac3_provisional_primary_tumor_files.csv"

STAR_ROOT = ROOT / "data/raw/cptac3/rnaseq_star_counts"

SCAN_OUTPUT = META / "112_cptac3_rna_simple_primary_technical_inventory.csv"
HANDOFF_112_PATH = META / "112_cptac3_qc_provenance_handoff.json"

In [3]:
# =============================================================================
# Load upstream data
# =============================================================================

with AUDIT_111_PATH.open(encoding="utf-8") as handle:
    audit_111 = json.load(handle)

index_111 = pd.read_csv(FILE_INDEX_111_PATH, dtype="string")
primary_111 = pd.read_csv(PRIMARY_111_PATH, dtype="string")

In [4]:
# =============================================================================
# Verify upstream integrity
# =============================================================================

if audit_111.get("full_payload_md5_verified") is not True:
    raise ValueError("Notebook 111 payload integrity is not verified")

expected_sha256 = audit_111["audit_artifacts"]["file_index"]["sha256"]

if calculate_sha256(FILE_INDEX_111_PATH) != expected_sha256:
    raise ValueError("Notebook 111 file index SHA-256 mismatch")

print("Notebook 111 source integrity: verified")

Notebook 111 source integrity: verified


In [5]:
# =============================================================================
# RNA-seq source inventory
# =============================================================================

idx_rna = index_111.loc[index_111["modality"].eq("rnaseq")].copy()

if len(idx_rna) != 2968:
    raise ValueError("Unexpected CPTAC-3 RNA-seq source universe")

print(f"Audited RNA-seq source files: {len(idx_rna):,}")

Audited RNA-seq source files: 2,968


## Primary-Tumor RNA-seq Source Candidates

Using the audited file, sample, and aliquot associations from Notebook 111, identify cases represented by exactly one primary-tumor STAR Counts file, with uniquely resolved sample and aliquot identities.

Cases with multiple primary samples or aliquots are characterized separately and excluded from the main technical inventory. No preferred-file selection or sequencing-QC eligibility decision is performed.

In [6]:
# =============================================================================
# Validate primary RNA-seq source identities
# =============================================================================

primary_rna = primary_111.loc[primary_111["modality"].eq("rnaseq")].copy()

required_columns = {
    "file_id",
    "case_uuid",
    "sample_uuid",
    "aliquot_uuid",
    "filename",
    "size_bytes",
    "provider_md5",
    "mapping_status",
    "sample_type",
    "primary_site",
    "disease_type",
}

missing = required_columns - set(primary_rna.columns)

if missing:
    raise ValueError(f"Missing primary RNA-seq columns: {sorted(missing)}")

identity_columns = ["file_id", "case_uuid", "sample_uuid", "aliquot_uuid"]

if primary_rna[identity_columns].isna().any().any():
    raise ValueError("Missing primary RNA-seq source identifiers")

if primary_rna["file_id"].duplicated().any():
    raise ValueError("Duplicated primary RNA-seq file identifiers")

if not primary_rna["mapping_status"].eq("unique_aliquot_sample").all():
    raise ValueError("Unresolved ancestry among primary RNA-seq candidates")

if not primary_rna["sample_type"].eq("Primary Tumor").all():
    raise ValueError("Unexpected sample type among primary RNA-seq candidates")

In [7]:
# =============================================================================
# Reconcile primary candidates with the audited source index
# =============================================================================

if idx_rna["file_id"].duplicated().any():
    raise ValueError("Duplicated source RNA-seq file identifiers")

matched = primary_rna[identity_columns].merge(
    idx_rna[identity_columns],
    on=identity_columns,
    how="inner",
    validate="one_to_one",
)

if len(matched) != len(primary_rna):
    raise ValueError("Primary RNA-seq identities differ from the source index")

print(f"Authenticated primary RNA-seq files: {len(primary_rna):,}")

Authenticated primary RNA-seq files: 1,731


In [8]:
# =============================================================================
# Classify primary-tumor case representation
# =============================================================================

case_structure = primary_rna.groupby("case_uuid").agg(
    n_files=("file_id", "nunique"),
    n_samples=("sample_uuid", "nunique"),
    n_aliquots=("aliquot_uuid", "nunique"),
)

simple_mask = case_structure[
    ["n_files", "n_samples", "n_aliquots"]
].eq(1).all(axis=1)

simple_case_ids = case_structure.index[simple_mask]

rna_e_simple_files = (
    primary_rna.loc[primary_rna["case_uuid"].isin(simple_case_ids)]
    .sort_values(["case_uuid", "file_id"], kind="stable")
    .reset_index(drop=True)
)

if (
    len(rna_e_simple_files) != len(simple_case_ids)
    or rna_e_simple_files["case_uuid"].duplicated().any()
):
    raise ValueError("Invalid uniquely represented primary RNA-seq cases")

print(f"Primary-tumor cases: {len(case_structure):,}")
print(f"Uniquely represented cases: {len(simple_case_ids):,}")
print(f"Cases with primary multiplicity: {(~simple_mask).sum():,}")

Primary-tumor cases: 1,370
Uniquely represented cases: 1,082
Cases with primary multiplicity: 288


In [9]:
# =============================================================================
# Characterize additional same-case RNA-seq files
# =============================================================================

source_counts = idx_rna.groupby("case_uuid").agg(
    n_rna_files=("file_id", "nunique"),
    n_ambiguous_rna_files=(
        "mapping_status",
        lambda values: int(values.eq("ambiguous_sample_ancestry").sum()),
    ),
)

case_ids = rna_e_simple_files["case_uuid"]

n_total = case_ids.map(source_counts["n_rna_files"])
n_ambiguous = case_ids.map(source_counts["n_ambiguous_rna_files"])

if n_total.isna().any() or n_ambiguous.isna().any():
    raise ValueError("Missing RNA-seq case-level source counts")

rna_e_simple_files["n_other_rna_files_in_case"] = (
    n_total.astype("int64") - 1
)

rna_e_simple_files["n_unresolved_ancestry_rna_files"] = (
    n_ambiguous.astype("int64")
)

if (
    rna_e_simple_files["n_other_rna_files_in_case"].lt(0).any()
    or (
        rna_e_simple_files["n_unresolved_ancestry_rna_files"]
        > rna_e_simple_files["n_other_rna_files_in_case"]
    ).any()
):
    raise ValueError("Inconsistent additional RNA-seq file counts")

In [10]:
# =============================================================================
# Summarize additional RNA-seq source representation
# =============================================================================

other = rna_e_simple_files["n_other_rna_files_in_case"]
ambiguous = rna_e_simple_files["n_unresolved_ancestry_rna_files"]

print(f"Cases with additional RNA files: {other.gt(0).sum():,}")
print(f"Additional RNA files: {other.sum():,}")
print(f"Cases with ambiguous-ancestry RNA: {ambiguous.gt(0).sum():,}")
print(f"Ambiguous-ancestry RNA files: {ambiguous.sum():,}")

Cases with additional RNA files: 530
Additional RNA files: 544
Cases with ambiguous-ancestry RNA: 37
Ambiguous-ancestry RNA files: 38


## STAR Counts Technical Characterization

Characterize the STAR Counts files corresponding to uniquely represented primary-tumor RNA-seq cases identified from Notebook 111.

For each file, assess gene identifier integrity, unstranded count validity, version-normalized Ensembl ID duplication, detected gene counts, and STAR special-category counts.

These measurements describe the technical representation of the source files. They do not establish independent sequencing-quality eligibility, expression comparability, or biological validity.

Malformed or inconsistent source files cause execution to stop. No empirical QC thresholds, sample exclusions, or alternative-file selection are applied.


In [11]:
# =============================================================================
# Load STAR Counts files
# =============================================================================

SPECIAL = ("N_unmapped", "N_multimapping", "N_noFeature", "N_ambiguous")


def load_star_counts(record):
    path = STAR_ROOT / str(record.file_id) / str(record.filename)

    if not path.is_file():
        raise FileNotFoundError(f"Missing STAR Counts file: {path}")

    if path.stat().st_size != int(record.size_bytes):
        raise ValueError(f"STAR Counts file size mismatch: {record.file_id}")

    return pd.read_csv(
        path,
        sep="\t",
        comment="#",
        usecols=["gene_id", "unstranded"],
        dtype={"gene_id": "string", "unstranded": "string"},
    )

In [12]:
# =============================================================================
# Characterize gene counts
# =============================================================================

def characterize_gene_counts(counts, file_id):
    if counts["gene_id"].isna().any():
        raise ValueError(f"Missing gene identifiers: {file_id}")

    special_mask = counts["gene_id"].str.startswith("N_", na=False)
    genes = counts.loc[~special_mask]

    if genes.empty or genes["gene_id"].duplicated().any():
        raise ValueError(f"Invalid STAR gene identifiers: {file_id}")

    versionless_ids = genes["gene_id"].str.replace(
        r"\.\d+$", "", regex=True
    )

    n_duplicate_ids = int(
        versionless_ids.value_counts(dropna=False).gt(1).sum()
    )

    values = pd.to_numeric(
        genes["unstranded"], errors="coerce"
    ).to_numpy(dtype="float64", na_value=np.nan)

    invalid = (
        ~np.isfinite(values)
        | (values < 0)
        | (values != np.floor(values))
    )

    valid_counts = not invalid.any()

    return {
        "n_gene_rows": len(genes),
        "n_duplicate_versionless_ids": n_duplicate_ids,
        "unstranded_nonnegative_integer_vector": bool(valid_counts),
        "n_invalid_unstranded_gene_values": int(invalid.sum()),
        "unstranded_gene_count_total": (
            float(values.sum()) if valid_counts else np.nan
        ),
        "n_detected_star_genes": (
            int((values > 0).sum()) if valid_counts else pd.NA
        ),
    }

In [13]:
# =============================================================================
# Characterize STAR special counts
# =============================================================================

def characterize_star_special_counts(counts):
    special_mask = counts["gene_id"].str.startswith("N_", na=False)

    result = {
        "n_star_special_rows": int(special_mask.sum()),
    }

    for special_id in SPECIAL:
        values = pd.to_numeric(
            counts.loc[
                counts["gene_id"].eq(special_id),
                "unstranded",
            ],
            errors="coerce",
        )

        result[f"star_{special_id.lower()}"] = (
            float(values.iloc[0])
            if len(values) == 1 and pd.notna(values.iloc[0])
            else np.nan
        )

    return result

In [14]:
# =============================================================================
# Build primary RNA-seq technical inventory
# =============================================================================

rows = []

for n, record in enumerate(rna_e_simple_files.itertuples(index=False), 1):
    counts = load_star_counts(record)

    rows.append(
        {
            "case_uuid": record.case_uuid,
            "file_id": record.file_id,
            "sample_uuid": record.sample_uuid,
            "aliquot_uuid": record.aliquot_uuid,
            "primary_site": record.primary_site,
            "disease_type": record.disease_type,
            **characterize_gene_counts(counts, record.file_id),
            **characterize_star_special_counts(counts),
            "n_other_rna_files_in_case": record.n_other_rna_files_in_case,
            "n_unresolved_ancestry_rna_files": (
                record.n_unresolved_ancestry_rna_files
            ),
        }
    )

    if n % 200 == 0 or n == len(rna_e_simple_files):
        print(f"STAR Counts processed: {n:,}/{len(rna_e_simple_files):,}")

rna_e_file_diagnostics = (
    pd.DataFrame(rows)
    .sort_values(["case_uuid", "file_id"], kind="stable")
    .reset_index(drop=True)
)

STAR Counts processed: 200/1,082
STAR Counts processed: 400/1,082
STAR Counts processed: 600/1,082
STAR Counts processed: 800/1,082
STAR Counts processed: 1,000/1,082
STAR Counts processed: 1,082/1,082


In [15]:
# =============================================================================
# Validate primary RNA-seq technical inventory
# =============================================================================

identity_columns = [
    "case_uuid",
    "file_id",
    "sample_uuid",
    "aliquot_uuid",
]

matched = rna_e_file_diagnostics[identity_columns].merge(
    rna_e_simple_files[identity_columns],
    on=identity_columns,
    validate="one_to_one",
)

if len(matched) != len(rna_e_simple_files):
    raise ValueError("STAR technical inventory identity mismatch")

if (
    not rna_e_file_diagnostics["unstranded_nonnegative_integer_vector"].all()
    or rna_e_file_diagnostics["n_invalid_unstranded_gene_values"].ne(0).any()
    or rna_e_file_diagnostics["n_duplicate_versionless_ids"].ne(0).any()
):
    raise ValueError("Invalid STAR Counts technical representation")


# Verify standard STAR special-category representation.
star_columns = [f"star_{name.lower()}" for name in SPECIAL]

missing_columns = set(star_columns) - set(rna_e_file_diagnostics.columns)

if missing_columns:
    raise ValueError(f"Missing STAR special columns: {sorted(missing_columns)}")

star_values = (
    rna_e_file_diagnostics[star_columns]
    .apply(pd.to_numeric, errors="coerce")
    .to_numpy(dtype="float64", na_value=np.nan)
)

if (
    not rna_e_file_diagnostics["n_star_special_rows"].eq(len(SPECIAL)).all()
    or not np.isfinite(star_values).all()
    or (star_values < 0).any()
    or (star_values != np.floor(star_values)).any()
):
    raise ValueError("Invalid STAR special-category count representation")

print(f"Source files characterized: {len(rna_e_file_diagnostics):,}")
print("Source identities: verified")
print("Unstranded count validity: verified")
print("Version-normalized gene identifiers: verified")
print("STAR special-category representation: verified")


Source files characterized: 1,082
Source identities: verified
Unstranded count validity: verified
Version-normalized gene identifiers: verified
STAR special-category representation: verified


In [16]:
# =============================================================================
# Summarize STAR Counts technical characteristics
# =============================================================================

summary_columns = [
    "n_gene_rows",
    "unstranded_gene_count_total",
    "n_detected_star_genes",
]

display(rna_e_file_diagnostics[summary_columns].describe().T)

print("STAR gene rows per file:")
print(sorted(rna_e_file_diagnostics["n_gene_rows"].unique().tolist()))

print("No empirical QC thresholds or sample exclusions applied")

,count,mean,std,min,25%,50%,75%,max
n_gene_rows,1082.0,6.066000e+04,0.000000e+00,60660.0,60660.0,60660.0,60660.00,60660.0
unstranded_gene_count_total,1082.0,3.883843e+07,8.633513e+06,1074810.0,33986743.5,38347562.0,43393965.25,84641397.0
n_detected_star_genes,1082.0,4.205390e+04,2.025331e+03,20953.0,41077.5,42264.0,43245.75,46782.0


STAR gene rows per file:
[60660]
No empirical QC thresholds or sample exclusions applied



## Additional RNA Files and Source Context

Characterize additional RNA-seq files associated with uniquely represented primary-tumor cases, preserving their sample-ancestry status and source specimen classification.

Distinguish files sharing the exact sample or aliquot identity from files associated only at the case level. Ambiguous ancestry is retained without assigning unsupported specimen relationships.

Additional files are not treated as alternative primary specimens or technical replicates. No source-file substitution or sequencing-QC exclusion is performed.


In [17]:
# =============================================================================
# Identify additional RNA-seq source files
# =============================================================================

rna_f_other_files = idx_rna.loc[
    idx_rna["case_uuid"].isin(rna_e_simple_files["case_uuid"])
    & ~idx_rna["file_id"].isin(rna_e_simple_files["file_id"])
].copy()

if rna_f_other_files[["case_uuid", "file_id"]].isna().any().any():
    raise ValueError("Missing additional RNA-seq source identifiers")

if rna_f_other_files["file_id"].duplicated().any():
    raise ValueError("Duplicated additional RNA-seq source files")

print(f"Additional RNA-seq files identified: {len(rna_f_other_files):,}")

Additional RNA-seq files identified: 544


In [18]:
# =============================================================================
# Verify additional RNA-seq source associations
# =============================================================================

observed_counts = rna_f_other_files.groupby("case_uuid").agg(
    n_extra_rna_files=("file_id", "size"),
    n_extra_ambiguous_ancestry=(
        "mapping_status",
        lambda values: int(
            values.eq("ambiguous_sample_ancestry").sum()
        ),
    ),
)

expected_counts = (
    rna_e_simple_files.set_index("case_uuid")[
        [
            "n_other_rna_files_in_case",
            "n_unresolved_ancestry_rna_files",
        ]
    ]
    .rename(
        columns={
            "n_other_rna_files_in_case": "n_extra_rna_files",
            "n_unresolved_ancestry_rna_files": (
                "n_extra_ambiguous_ancestry"
            ),
        }
    )
    .astype("int64")
)

observed_counts = observed_counts.reindex(
    expected_counts.index, fill_value=0
)

unexpected_cases = observed_counts.index.difference(expected_counts.index)

if len(unexpected_cases) > 0:
    raise ValueError("Unexpected cases among additional RNA-seq files")

observed_counts = (
    observed_counts.reindex(
        index=expected_counts.index,
        columns=expected_counts.columns,
        fill_value=0,
    )
    .astype("int64")
)

if not observed_counts.equals(expected_counts):
    raise ValueError("Additional RNA-seq source counts do not reconcile")

print("Additional RNA-seq associations: verified")

Additional RNA-seq associations: verified


In [19]:
# =============================================================================
# Characterize case-level RNA-seq source context
# =============================================================================

rna_f_case_context = rna_e_simple_files[
    [
        "case_uuid",
        "file_id",
        "primary_site",
        "disease_type",
        "n_other_rna_files_in_case",
        "n_unresolved_ancestry_rna_files",
    ]
].rename(
    columns={
        "n_other_rna_files_in_case": "n_extra_rna_files",
        "n_unresolved_ancestry_rna_files": (
            "n_extra_ambiguous_ancestry"
        ),
    }
).copy()

rna_f_case_context["source_file_status"] = np.select(
    [
        rna_f_case_context["n_extra_ambiguous_ancestry"].gt(0),
        rna_f_case_context["n_extra_rna_files"].gt(0),
    ],
    [
        "OTHER_FILE_INCLUDES_AMBIGUOUS_ANCESTRY",
        "OTHER_SOURCE_RNA_FILES_NO_AMBIGUOUS_ANCESTRY",
    ],
    default="NO_OTHER_SOURCE_RNA_FILES",
)

display(
    rna_f_case_context["source_file_status"]
    .value_counts()
    .rename_axis("source_file_status")
    .reset_index(name="n_cases")
)

,source_file_status,n_cases
0,NO_OTHER_SOURCE_RNA_FILES,552
1,OTHER_SOURCE_RNA_FILES_NO_AMBIGUOUS_ANCESTRY,493
2,OTHER_FILE_INCLUDES_AMBIGUOUS_ANCESTRY,37


In [20]:
# =============================================================================
# Identify exact specimen relationships
# =============================================================================

primary_ids = rna_e_simple_files.set_index("case_uuid")[
    ["sample_uuid", "aliquot_uuid"]
]

if not primary_ids.index.is_unique:
    raise ValueError("Duplicated primary case identifiers")

resolved = (
    rna_f_other_files["mapping_status"]
    .eq("unique_aliquot_sample")
    .fillna(False)
)

primary_samples = rna_f_other_files["case_uuid"].map(
    primary_ids["sample_uuid"]
)

primary_aliquots = rna_f_other_files["case_uuid"].map(
    primary_ids["aliquot_uuid"]
)

rna_f_other_files["same_exact_sample_as_primary"] = (
    resolved
    & rna_f_other_files["sample_uuid"].eq(primary_samples).fillna(False)
).to_numpy(dtype=bool, na_value=False)

rna_f_other_files["same_exact_aliquot_as_primary"] = (
    resolved
    & rna_f_other_files["aliquot_uuid"].eq(primary_aliquots).fillna(False)
).to_numpy(dtype=bool, na_value=False)

In [21]:
# =============================================================================
# Classify additional RNA-seq specimen context
# =============================================================================

status = rna_f_other_files["mapping_status"].astype("string")

sample_type = (
    rna_f_other_files["sample_type"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

ambiguous = status.eq("ambiguous_sample_ancestry").fillna(False)
resolved = status.eq("unique_aliquot_sample").fillna(False)

normal = (
    sample_type.eq("normal")
    | sample_type.str.endswith(" normal")
).fillna(False)

conditions = [
    ambiguous,
    ~resolved,
    sample_type.eq("primary tumor").fillna(False),
    normal,
    sample_type.eq("recurrent tumor").fillna(False),
    sample_type.eq("metastatic").fillna(False),
]

classes = [
    "ANCESTRY_AMBIGUOUS_NO_SAMPLE_ATTRIBUTION",
    "OTHER_MAPPING_UNRESOLVED",
    "RESOLVED_PRIMARY_TUMOR_REVIEW",
    "RESOLVED_NORMAL_TISSUE",
    "RESOLVED_RECURRENT_TUMOR",
    "RESOLVED_METASTATIC_TUMOR",
]

rna_f_other_files["other_rna_source_class"] = np.select(
    [condition.to_numpy(dtype=bool) for condition in conditions],
    classes,
    default="RESOLVED_OTHER_OR_UNSPECIFIED_SAMPLE_TYPE",
)

In [22]:
# =============================================================================
# Validate and summarize additional specimen context
# =============================================================================

same_sample = rna_f_other_files["same_exact_sample_as_primary"]
same_aliquot = rna_f_other_files["same_exact_aliquot_as_primary"]

if (same_aliquot & ~same_sample).any():
    raise ValueError("Inconsistent sample and aliquot relationships")

unresolved = ~resolved

if rna_f_other_files.loc[
    unresolved,
    ["same_exact_sample_as_primary", "same_exact_aliquot_as_primary"],
].any().any():
    raise ValueError("Unresolved ancestry assigned to an exact specimen")

source_summary = (
    rna_f_other_files.groupby("other_rna_source_class")
    .agg(
        source_files=("file_id", "nunique"),
        distinct_cases=("case_uuid", "nunique"),
        same_sample_files=("same_exact_sample_as_primary", "sum"),
        same_aliquot_files=("same_exact_aliquot_as_primary", "sum"),
    )
    .reset_index()
    .sort_values("source_files", ascending=False)
    .reset_index(drop=True)
)

if source_summary["source_files"].sum() != len(rna_f_other_files):
    raise ValueError("Additional RNA-seq source classification mismatch")

display(source_summary)

,other_rna_source_class,source_files,distinct_cases,same_sample_files,same_aliquot_files
0,RESOLVED_NORMAL_TISSUE,488,486,0,0
1,ANCESTRY_AMBIGUOUS_NO_SAMPLE_ATTRIBUTION,38,37,0,0
2,RESOLVED_RECURRENT_TUMOR,18,16,0,0


## Downstream Technical Outputs

Persist the validated STAR Counts technical inventory and its provenance manifest for use by Notebook 113.

The inventory preserves the source identities and technical diagnostics of uniquely represented primary-tumor RNA-seq cases. The manifest records upstream source fingerprints, output integrity, and the boundaries of the technical characterization.

In [23]:
# =============================================================================
# Persist STAR Counts technical inventory
# =============================================================================

required_columns = [
    "case_uuid",
    "file_id",
    "sample_uuid",
    "aliquot_uuid",
    "n_gene_rows",
    "n_duplicate_versionless_ids",
    "unstranded_nonnegative_integer_vector",
    "n_invalid_unstranded_gene_values",
    "unstranded_gene_count_total",
    "n_detected_star_genes",
]

missing = set(required_columns) - set(rna_e_file_diagnostics.columns)

if missing:
    raise ValueError(f"Missing technical inventory columns: {sorted(missing)}")

# Persist the validated technical inventory.
SCAN_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
rna_e_file_diagnostics.to_csv(SCAN_OUTPUT, index=False)

# Verify the persisted schema, row count, and source identities.
persisted = pd.read_csv(SCAN_OUTPUT, dtype="string")

if (
    len(persisted) != len(rna_e_simple_files)
    or list(persisted.columns) != list(rna_e_file_diagnostics.columns)
):
    raise ValueError("Persisted technical inventory structure mismatch")

pd.testing.assert_frame_equal(
    persisted[identity_columns],
    rna_e_file_diagnostics[identity_columns].astype("string"),
    check_dtype=False,
)

inventory_sha256 = calculate_sha256(SCAN_OUTPUT)

print(f"Technical inventory: {project_relative_path(SCAN_OUTPUT)}")
print(f"Source files: {len(persisted):,}")
print(f"SHA-256: {inventory_sha256}")

Technical inventory: data/interim/metadata/112_cptac3_rna_simple_primary_technical_inventory.csv
Source files: 1,082
SHA-256: 0e3ad06b960f6dc4aa11e523db23948ff6763bdb97e8f36f959caebca104607c


In [24]:
# =============================================================================
# Write the downstream technical inventory provenance manifest
# =============================================================================

# Authenticate the source artifacts generated by Notebook 111.
source_inputs = {
    "acquisition_audit": AUDIT_111_PATH,
    "file_index": FILE_INDEX_111_PATH,
    "provisional_primary_files": PRIMARY_111_PATH,
}

input_provenance = {
    name: {
        "path": project_relative_path(path),
        "sha256": calculate_sha256(path),
    }
    for name, path in source_inputs.items()
}

# Verify that the technical inventory has not changed since persistence.
if calculate_sha256(SCAN_OUTPUT) != inventory_sha256:
    raise ValueError("Technical inventory fingerprint changed")

# Record the observed source scope without assigning QC eligibility.
source_scope = {
    "audited_rna_files": int(len(idx_rna)),
    "provisional_primary_files": int(len(primary_rna)),
    "provisional_primary_cases": int(len(case_structure)),
    "simple_primary_cases": int(len(rna_e_simple_files)),
    "primary_multiplicity_cases": int(
        len(case_structure) - len(rna_e_simple_files)
    ),
    "additional_rna_files": int(len(rna_f_other_files)),
    "additional_ambiguous_ancestry_files": int(
        rna_f_other_files["mapping_status"]
        .eq("ambiguous_sample_ancestry")
        .sum()
    ),
}

if len(rna_e_file_diagnostics) != source_scope["simple_primary_cases"]:
    raise ValueError("Technical inventory and source scope disagree")

handoff_112 = {
    "schema_version": 1,
    "notebook": 112,
    "role": "RNA_SOURCE_TECHNICAL_CHARACTERIZATION",
    "data_access_scope": "AUDITED_NOTEBOOK_111_INPUTS_AND_LOCAL_STAR_COUNTS",
    "software": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
    },
    "inputs": input_provenance,
    "source_scope": source_scope,
    "technical_characterization": {
        "count_column": "unstranded",
        "source_files_characterized": int(len(rna_e_file_diagnostics)),
        "independent_sequencing_qc_performed": False,
        "empirical_qc_thresholds_applied": False,
        "qc_based_exclusions_applied": False,
    },
    "scientific_limitations": [
        "Independent sequencing and library QC were not assessed",
        "STAR assigned-gene counts do not establish sequencing depth",
        "Library and batch comparability were not established",
        "Tumor purity and proliferation confounding were not assessed",
        "Primary specimen multiplicity was not adjudicated",
        "Lineage eligibility and donor independence were not established",
    ],
    "source_technical_audit_complete": True,
    "QC_eligibility_authorized": False,
    "cohort_frozen": False,
    "program_scores_computed": False,
    "biological_replication_performed": False,
    "outputs": {
        "technical_inventory": {
            "path": project_relative_path(SCAN_OUTPUT),
            "rows": int(len(rna_e_file_diagnostics)),
            "sha256": inventory_sha256,
        },
    },
}

# Persist and verify the complete manifest.
HANDOFF_112_PATH.parent.mkdir(parents=True, exist_ok=True)

with HANDOFF_112_PATH.open("w", encoding="utf-8") as handle:
    json.dump(handoff_112, handle, indent=2, ensure_ascii=False)
    handle.write("\n")

with HANDOFF_112_PATH.open(encoding="utf-8") as handle:
    persisted_handoff = json.load(handle)

if persisted_handoff != handoff_112:
    raise ValueError("Persisted technical handoff differs from source data")

print("Handoff:", project_relative_path(HANDOFF_112_PATH))
print("Technical inventory rows:", source_scope["simple_primary_cases"])
print("Technical inventory SHA-256:", inventory_sha256)

Handoff: data/interim/metadata/112_cptac3_qc_provenance_handoff.json
Technical inventory rows: 1082
Technical inventory SHA-256: 0e3ad06b960f6dc4aa11e523db23948ff6763bdb97e8f36f959caebca104607c



## Results and Scientific Limitations

The audited CPTAC-3 RNA-seq source inventory contains 1,731 provisionally primary-tumor files representing 1,370 cases. Of these, 1,082 cases have exactly one primary-tumor STAR Counts file, sample, and aliquot. The remaining 288 cases have primary-source multiplicity and are not included in the main technical inventory; no alternative specimen selection or multiplicity adjudication is performed.

All 1,082 uniquely represented primary-tumor STAR Counts files were characterized. The technical assessment verified gene identifier integrity, nonnegative integer `unstranded` counts, version-normalized gene identifier uniqueness, and STAR special-category representation. Each file contained 60,660 gene rows. No structural inconsistencies were detected in the assessed count representations.

Additional RNA-seq source records were characterized without assigning unsupported specimen relationships. Among the uniquely represented primary-tumor cases, 530 have additional RNA-seq files (544 files in total), including 37 cases with ambiguous sample ancestry (38 files). These additional records were not substituted for the primary source files.

The technical inventory and its provenance manifest were written to `data/interim/metadata/`. The manifest records the audited Notebook 111 input fingerprints, source scope, technical inventory row count, and SHA-256 identity required by Notebook 113.

**Scientific limitations:** The technical assessment establishes consistency of the available STAR count representations, not independent sequencing or library quality. Read-level sequencing QC, RNA integrity, library complexity, batch comparability, tumor purity, and proliferation confounding were not independently evaluated. Assigned-gene counts and STAR special-category tallies are descriptive and do not establish sequencing-quality acceptance.

The restriction to uniquely represented primary-tumor cases is structural, not based on empirical sequencing-QC thresholds. No sequencing-quality PASS/FAIL classification or QC-based sample exclusion was performed. Cancer-lineage harmonization, primary-source multiplicity adjudication, TCGA–CPTAC donor independence, and external program replication remain outside the scope of Notebook 112.

The resulting artifacts provide an authenticated technical input for downstream cohort construction, without establishing analytic eligibility or biological replication.
